# Gradient Boosting Model: XGBoost (Label Encoded & Diagnosis Embeddings)

### Settings

In [1]:
# Settings

import os
import json
import pandas as pd
import numpy as np
import torch
import pickle
import optuna
import joblib
from pathlib import Path
from numpy import f2py
from sklearn.decomposition import PCA
from copy import deepcopy
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, fbeta_score, f1_score, recall_score, precision_score, precision_recall_curve, auc, average_precision_score
from src.utils.data_fetch import DataLoader
from src.utils.model_eval.evaluation import evaluate_predictions
from src.visualisations.viz_model import plot_confusion_matrix


SEED = 1234
TUNE = True
TUNE_PROB = True
TUNE_SAVE = True
TUNE_METRIC = 'prauc' # recall/auc/f1/f2/prauc
TUNE_OUTPUT = f'xgboost_label_emb_params_{TUNE_METRIC}.json'
TUNE_PROB_OUTPUT = f'xgboost_label_emb_prob_{TUNE_METRIC}.json'
MODEL_OUTPUT = f'xgboost_label_emb_model_{TUNE_METRIC}.json'
MODEL_SAVE = True
MODEL_LOAD = False
N_TRIALS = 150


PATH_EMBEDDINGS = Path('../../data/embedding/diag.pickle')
PATH_SHAP_TRAIN = Path('../../data/shap/xgboost_label_shap_train.csv')
PATH_SHAP_VAL = Path('../../data/shap/xgboost_label_shap_val.csv')
PATH_SHAP_TEST = Path('../../data/shap/xgboost_label_shap_test.csv')

D:\Projects\datasci_207_project\ds207_final_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load data

In [2]:
# Load data
X_train, X_val, X_test, y_train, y_val, y_test = DataLoader().fetch_data(version='final_onehot')
# X_train, X_val, X_test, y_train, y_val, y_test = DataLoader().fetch_data(version='interim')
X_train_interim, X_val_interim, X_test_interim, _, _, _ = DataLoader().fetch_data(version='interim')

# Remove ID and drug (not grouped) columns
id_cols = [c for c in X_train.columns if 'encounter_id' in c or 'patient_nbr' in c]
diag_cols = [c for c in X_train.columns if 'diag_' in c]
drug_cols = [c for c in X_train.columns if 'prescript' in c and (('change' not in c) and ('prescribed' not in c))]
drug_group_cols = [c for c in X_train.columns if 'prescript' in c and (('prescribed' in c) or ('change' in c))]
drop_cols = id_cols# + drug_cols + drug_group_cols
X_train = X_train.drop(drop_cols, axis=1)
X_val = X_val.drop(drop_cols, axis=1)
X_test = X_test.drop(drop_cols, axis=1)

# X_train = X_train.replace('?', 'Unknown')
# X_val = X_val.replace('?', 'Unknown')
# X_test = X_test.replace('?', 'Unknown')

# Convert categorical column types
# cat_cols = X_train.select_dtypes(include=['object', 'string']).columns
# num_cols = [c for c in X_train.columns if c not in cat_cols]
num_cols = [
    'pipeline_standard__time_in_hospital',
    'pipeline_standard__num_lab_procedures',
    'pipeline_standard__num_procedures',
    'pipeline_standard__num_medications',
    'pipeline_standard__number_outpatient',
    'pipeline_standard__number_emergency',
    'pipeline_standard__number_inpatient',
    'pipeline_standard__number_diagnoses'
] + drug_group_cols
cat_cols = X_train.columns.difference(num_cols).tolist()
for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# X_train = X_train[num_cols]
# X_val = X_val[num_cols]
# X_test = X_test[num_cols]

# for x in [X_train, X_val, X_test]:
#     x['total_visits'] = x['number_inpatient'] + x['number_outpatient'] + x['number_emergency']
#     x['num_med_per_diag'] = x['num_medications'] / x['number_diagnoses'].apply(lambda x: max(x, 1))
#     x['num_med_per_proc'] = x['num_medications'] / x['num_procedures'].apply(lambda x: max(x, 1))
#     x['time_x_diag'] = x['time_in_hospital'].apply(lambda x: max(x, 1)) * x['number_diagnoses'].apply(lambda x: max(x, 1))


# Modify special characters in column names for xgboost tuning
cols_update = [c.replace('[', '(').replace(')', ')').replace('<', 'lt').replace('>', 'gt') for c in X_train.columns]
X_train.columns = cols_update
X_val.columns = cols_update
X_test.columns = cols_update
#
# X_train_t = torch.tensor(X_train.values, dtype=torch.float32, device='cuda')
# X_val_t = torch.tensor(X_val.values, dtype=torch.float32, device='cuda')
# y_train_t = torch.tensor(y_train.values, dtype=torch.float32, device='cuda')
# y_val_t = torch.tensor(y_val.values, dtype=torch.float32, device='cuda')

Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/onehot/X_train_mini.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/onehot/X_val.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/onehot/X_test.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/onehot/y_train_mini.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/onehot/y_val.csv 
Data loaded from path:  https://raw.githubusercontent.com/wip-0/ds207_final_project/refs/heads/main/data/final_processed/base_kf/onehot/y_test.csv 
Data imported succesfully from paths
Data loaded from path:  https://raw.githubusercontent.com/wip-0/d

### Preprocess embeddings

In [3]:
# Load embedding dictionary
with open(PATH_EMBEDDINGS, 'rb') as f:
    dict_emb = pickle.load(f)

# Load SHAP
# df_shap_train = pd.read_csv(PATH_SHAP_TRAIN, index_col=0)
# df_shap_val = pd.read_csv(PATH_SHAP_VAL, index_col=0)
# df_shap_test = pd.read_csv(PATH_SHAP_TEST, index_col=0)
#
# # Create mask of valid diagnosis codes
# mask_train = X_train_interim[['diag_1', 'diag_2', 'diag_3']].isin(['?', np.nan])
# mask_val = X_val_interim[['diag_1', 'diag_2', 'diag_3']].isin(['?', np.nan])
# mask_test = X_test_interim[['diag_1', 'diag_2', 'diag_3']].isin(['?', np.nan])
# mask_train.columns = ['diag_1_shap', 'diag_2_shap', 'diag_3_shap']
# mask_val.columns = ['diag_1_shap', 'diag_2_shap', 'diag_3_shap']
# mask_test.columns = ['diag_1_shap', 'diag_2_shap', 'diag_3_shap']
#
# # Set invalid code weight to 0
# df_shap_train[mask_train] = 0
# df_shap_val[mask_val] = 0
# df_shap_test[mask_test] = 0
#
# # Get weight
# df_wgh_train = df_shap_train.abs().div(df_shap_train.abs().sum(axis=1), axis=0)
# df_wgh_val = df_shap_val.abs().div(df_shap_val.abs().sum(axis=1), axis=0)
# df_wgh_test = df_shap_test.abs().div(df_shap_test.abs().sum(axis=1), axis=0)
#
# emb_diag_train = (
# dict_emb['diag_1']['train'].fillna(0).mul(df_wgh_train['diag_1_shap'], axis=0).values +
# dict_emb['diag_2']['train'].fillna(0).mul(df_wgh_train['diag_2_shap'], axis=0).values +
# dict_emb['diag_3']['train'].fillna(0).mul(df_wgh_train['diag_3_shap'], axis=0).values
# )
# emb_diag_val = (
#     dict_emb['diag_1']['val'].fillna(0).mul(df_wgh_val['diag_1_shap'], axis=0).values +
#     dict_emb['diag_2']['val'].fillna(0).mul(df_wgh_val['diag_2_shap'], axis=0).values +
#     dict_emb['diag_3']['val'].fillna(0).mul(df_wgh_val['diag_3_shap'], axis=0).values
# )
# emb_diag_test = (
#     dict_emb['diag_1']['test'].fillna(0).mul(df_wgh_test['diag_1_shap'], axis=0).values +
#     dict_emb['diag_2']['test'].fillna(0).mul(df_wgh_test['diag_2_shap'], axis=0).values +
#     dict_emb['diag_3']['test'].fillna(0).mul(df_wgh_test['diag_3_shap'], axis=0).values
# )

emb_diag_1_train = dict_emb['diag_1']['train'].fillna(0)
emb_diag_1_val   = dict_emb['diag_1']['val'].fillna(0)
emb_diag_1_test  = dict_emb['diag_1']['test'].fillna(0)

emb_diag_2_train = dict_emb['diag_2']['train'].fillna(0)
emb_diag_2_val   = dict_emb['diag_2']['val'].fillna(0)
emb_diag_2_test  = dict_emb['diag_2']['test'].fillna(0)

emb_diag_3_train = dict_emb['diag_3']['train'].fillna(0)
emb_diag_3_val   = dict_emb['diag_3']['val'].fillna(0)
emb_diag_3_test  = dict_emb['diag_3']['test'].fillna(0)

dim = emb_diag_1_train.shape[1]

df_emb_diag_1_train = pd.DataFrame(emb_diag_1_train)
df_emb_diag_1_val   = pd.DataFrame(emb_diag_1_val  )
df_emb_diag_1_test  = pd.DataFrame(emb_diag_1_test )
df_emb_diag_2_train = pd.DataFrame(emb_diag_2_train)
df_emb_diag_2_val   = pd.DataFrame(emb_diag_2_val  )
df_emb_diag_2_test  = pd.DataFrame(emb_diag_2_test )
df_emb_diag_3_train = pd.DataFrame(emb_diag_3_train)
df_emb_diag_3_val   = pd.DataFrame(emb_diag_3_val  )
df_emb_diag_3_test  = pd.DataFrame(emb_diag_3_test )

# df_emb_diag_train = pd.concat([df_emb_diag_1_train, df_emb_diag_2_train, df_emb_diag_3_train], axis = 1)
# df_emb_diag_val   = pd.concat([df_emb_diag_1_val  , df_emb_diag_2_val  , df_emb_diag_3_val]  , axis = 1)
# df_emb_diag_test  = pd.concat([df_emb_diag_1_test , df_emb_diag_2_test , df_emb_diag_3_test] , axis = 1)

# Dimensionality reduction by PCA
n_components = 64
pca   = PCA(n_components=n_components)
pca_1 = PCA(n_components=n_components)
pca_2 = PCA(n_components=n_components)
pca_3 = PCA(n_components=n_components)

df_emb_diag_1_train = pd.DataFrame(pca_1.fit_transform(df_emb_diag_1_train), columns=['diag_emb1_' + str(i) for i in range(n_components)])
df_emb_diag_1_val   = pd.DataFrame(pca_1.transform(df_emb_diag_1_val), columns=['diag_emb1_' + str(i) for i in range(n_components)])
df_emb_diag_1_test  = pd.DataFrame(pca_1.transform(df_emb_diag_1_test), columns=['diag_emb1_' + str(i) for i in range(n_components)])

df_emb_diag_2_train = pd.DataFrame(pca_2.fit_transform(df_emb_diag_2_train), columns=['diag_emb2_' + str(i) for i in range(n_components)])
df_emb_diag_2_val   = pd.DataFrame(pca_2.transform(df_emb_diag_2_val), columns=['diag_emb2_' + str(i) for i in range(n_components)])
df_emb_diag_2_test  = pd.DataFrame(pca_2.transform(df_emb_diag_2_test), columns=['diag_emb2_' + str(i) for i in range(n_components)])

df_emb_diag_3_train = pd.DataFrame(pca_3.fit_transform(df_emb_diag_3_train), columns=['diag_emb3_' + str(i) for i in range(n_components)])
df_emb_diag_3_val   = pd.DataFrame(pca_3.transform(df_emb_diag_3_val), columns=['diag_emb3_' + str(i) for i in range(n_components)])
df_emb_diag_3_test  = pd.DataFrame(pca_3.transform(df_emb_diag_3_test), columns=['diag_emb3_' + str(i) for i in range(n_components)])

df_emb_diag_train = pd.concat([df_emb_diag_1_train, df_emb_diag_2_train, df_emb_diag_3_train], axis = 1)
df_emb_diag_val   = pd.concat([df_emb_diag_1_val  , df_emb_diag_2_val  , df_emb_diag_3_val]  , axis = 1)
df_emb_diag_test  = pd.concat([df_emb_diag_1_test , df_emb_diag_2_test , df_emb_diag_3_test] , axis = 1)

X_train = pd.concat([X_train, df_emb_diag_train], axis = 1)
X_val   = pd.concat([X_val  , df_emb_diag_val]  , axis = 1)
X_test  = pd.concat([X_test , df_emb_diag_test] , axis = 1)

print(pca_1.explained_variance_ratio_.cumsum())
print(pca_2.explained_variance_ratio_.cumsum())
print(pca_3.explained_variance_ratio_.cumsum())

#
# emb_diag_1_train = pca.fit_transform(emb_diag_1_train)
# emb_diag_1_val   = pca.transform(emb_diag_1_val)
# emb_diag_1_test  = pca.transform(emb_diag_1_test)
#
# emb_diag_2_train = pca.fit_transform(emb_diag_2_train)
# emb_diag_2_val   = pca.transform(emb_diag_2_val)
# emb_diag_2_test  = pca.transform(emb_diag_2_test)
#
# emb_diag_3_train = pca.fit_transform(emb_diag_3_train)
# emb_diag_3_val   = pca.transform(emb_diag_3_val)
# emb_diag_3_test  = pca.transform(emb_diag_3_test)
#
# df_emb_diag_train = pd.DataFrame(emb_diag_train, columns=['diag_emb_' + str(i) for i in range(emb_diag_train.shape[1])])
# df_emb_diag_val = pd.DataFrame(emb_diag_val, columns=['diag_emb_' + str(i) for i in range(emb_diag_val.shape[1])])
# df_emb_diag_test = pd.DataFrame(emb_diag_test, columns=['diag_emb_' + str(i) for i in range(emb_diag_test.shape[1])])


# Combine dataframes
# X_train = pd.concat([X_train, df_emb_diag_train], axis=1)
# X_val = pd.concat([X_val, df_emb_diag_val], axis=1)
# X_test = pd.concat([X_test, df_emb_diag_test], axis=1)

[0.12647372 0.21753171 0.28102666 0.33691587 0.38870996 0.43856723
 0.48169658 0.51771049 0.54814368 0.57674385 0.60401841 0.62834914
 0.65164207 0.67177341 0.69012906 0.70818752 0.72435049 0.73991688
 0.75426771 0.76683124 0.77882551 0.78966905 0.79960987 0.80921284
 0.81807968 0.82639372 0.83393975 0.8412497  0.847998   0.85408395
 0.85981873 0.86546537 0.87080225 0.87592725 0.88095149 0.88542467
 0.88985493 0.89390704 0.89761756 0.90118373 0.90454564 0.90767668
 0.91074507 0.91369149 0.91661445 0.91930328 0.92190307 0.92433633
 0.92670695 0.92898684 0.93117873 0.93320036 0.93514855 0.93694717
 0.93871633 0.94043305 0.94208329 0.94367975 0.94521497 0.94665972
 0.94802395 0.94934918 0.95062903 0.95184836]
[0.12620151 0.23234555 0.3146341  0.37810194 0.43846587 0.48870301
 0.52773885 0.56108148 0.59044942 0.61875066 0.64585199 0.66955704
 0.6916684  0.71068505 0.72743211 0.74298252 0.75791391 0.77143187
 0.78345824 0.79372776 0.80379815 0.81361909 0.82234142 0.83054116
 0.83807298 0.84

### Hyperparameter tuning

In [4]:
# Define objective function
fixed_params = {
    'objective'         : 'binary:logistic',
    'eval_metric'       : 'logloss',
    'tree_method'       : 'hist',
    'device'            : 'cuda',
    'enable_categorical': True,
    'random_state'      : SEED,
    'n_jobs'            : -1,
}

def objective(trial):
    """
    Hyperparameter tuning objective function.

    Args:
        trial: Tuning hyperparameter trial

    Returns:
        Assessment score
    """
    params = {
        # 'n_estimators'  : trial.suggest_int('n_estimators'      , 300  , 2000),
        'n_estimators'    : 6000,
        'max_depth'       : trial.suggest_int('max_depth'         , 6    , 12),
        'min_child_weight': trial.suggest_float('min_child_weight', 0.1  , 100  , log = True),
        'subsample'       : trial.suggest_float('subsample'       , 0.2  , 1.0),
        'learning_rate'   : trial.suggest_float('learning_rate'   , 0.005, 0.2  , log = True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.2  , 1.0),
        'gamma'           : trial.suggest_float('gamma'           , 0    , 10),
        'reg_alpha'       : trial.suggest_float('reg_alpha'       , 0.001, 10   , log = True),
        'reg_lambda'      : trial.suggest_float('reg_lambda'      , 0.001, 10   , log = True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.8  , 1.2),
    }
    params.update(fixed_params)

    # Fit the model
    model = XGBClassifier(**params, early_stopping_rounds=50)
    model.fit(X_train,
              y_train,
              eval_set=[(X_val, y_val)],
              verbose=False
              )
    trial.set_user_attr('best_iteration', model.best_iteration)

    y_prob_val = model.predict_proba(X_val)[:, 1]
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)

    acc_train = accuracy_score(y_train, y_pred_train)
    acc_val = accuracy_score(y_val, y_pred_val)
    if acc_train - acc_val > 0.10:  # >10% gap suggests overfitting
        print(f"Warning: Possible overfitting")
        return 0

    recall = recall_score(y_val, y_pred_val)
    precision = precision_score(y_val, y_pred_val)
    accuracy = accuracy_score(y_val, y_pred_val)
    roc_auc = roc_auc_score(y_val, y_prob_val)

    match TUNE_METRIC:
        case 'recall':
            score = recall
        case 'f1':
            score = f1_score(y_val, y_pred_val, average='macro')
        case 'f2':
            score = fbeta_score(y_val, y_pred_val, beta=2, average='macro')
        case 'auc':
            score = roc_auc_score(y_val, y_prob_val)
        case 'prauc':
            # p, r, th = precision_recall_curve(y_val, y_prob_val)
            # score = auc(r, p)
            score = average_precision_score(y_val, y_prob_val)

    trial_params = trial.params
    trial_params['n_estimators'] = model.best_iteration
    print(f'Trial: {trial.number} - Score: {score} | Recall: {recall} | Precision: {precision} | Accuracy: {accuracy} | ROC-AUC: {roc_auc} | Parameters: {trial_params}')

    return score


def log_callback(study, trial):
    """
    Optuner logging callback.
    """
    print(f'Best trial: {study.best_trial.number} | Score: {study.best_value}\n')

In [ ]:
# Perform tuning and save the best parameters
if TUNE:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=-1, callbacks=[log_callback])
    best_params = study.best_params
    best_params['n_estimators'] = study.best_trial.user_attrs.get('best_iteration') + 1
    print('Best Trial:', study.best_trial.number)
    print('Best Value:', study.best_value)
    print('Best Parameters:', best_params)

    if TUNE_SAVE:
        # Save output
        print('Parameters saved to:', TUNE_OUTPUT)
        with open(os.path.join(os.getcwd(), TUNE_OUTPUT), 'w') as f:
            json.dump(best_params, f)

Trial: 7 - Score: 0.6541262133313965 | Recall: 0.5738545338032682 | Precision: 0.6150412087912088 | Accuracy: 0.6299260154008758 | ROC-AUC: 0.6815998289600778 | Parameters: {'max_depth': 7, 'min_child_weight': 0.6632422187565378, 'subsample': 0.9411670665042617, 'learning_rate': 0.18201703161238447, 'colsample_bytree': 0.9767209912989956, 'gamma': 4.924707007683859, 'reg_alpha': 0.05074490495411771, 'reg_lambda': 0.7425605460410852, 'scale_pos_weight': 1.083607441042744, 'n_estimators': 35}
Best trial: 7 | Score: 0.6541262133313965

Best trial: 7 | Score: 0.6541262133313965

Trial: 13 - Score: 0.6572917571645518 | Recall: 0.5006942219374132 | Precision: 0.6492175598947514 | Accuracy: 0.6372238159947657 | ROC-AUC: 0.6866103365782407 | Parameters: {'max_depth': 7, 'min_child_weight': 0.172569360101611, 'subsample': 0.7325365784826523, 'learning_rate': 0.17522115228257218, 'colsample_bytree': 0.9669957204556772, 'gamma': 7.510762420162752, 'reg_alpha': 0.029643682943470234, 'reg_lambda': 

### Probability threshold tuning

In [ ]:
def tune_prob_th(best_params, min_recall=0.8):
    """
    Probability threshold tuning by maximizing precision for a given recall threshold.
    """
    final_params = deepcopy(fixed_params)
    final_params.update(best_params)
    print('Final params:', final_params)

    # Fit the model
    model = XGBClassifier(**final_params)
    model.fit(X_train,
              y_train,
              )
    y_prob = model.predict_proba(X_val)[:, 1]
    precision, recall, th = precision_recall_curve(y_val, y_prob)

    # remove extra endpoint w/o th
    precision = precision[:-1]
    recall = recall[:-1]

    # Get thresholds above min recall
    idx_r = np.flatnonzero(recall > min_recall)
    max_p = np.max(precision[idx_r])
    idx_best_th = idx_r[np.isclose(precision[idx_r], max_p, rtol=1e-10, atol=1e-10)]
    best_th = th[idx_best_th[np.argmax(th[idx_best_th])]]

    return float(best_th)

# Tune probability threshold
if TUNE_PROB:

    with open(os.path.join(os.getcwd(), TUNE_OUTPUT), 'r') as f:
        best_params = json.load(f)

    best_th = tune_prob_th(best_params)

    if TUNE_SAVE:
        # Save output
        print('Parameters saved to:', TUNE_PROB_OUTPUT)
        with open(os.path.join(os.getcwd(), TUNE_PROB_OUTPUT), 'w') as f:
            json.dump({'best_th': best_th}, f)

### Model training with the best parameters

In [ ]:
# Fit model
if not MODEL_LOAD:

    # Load parameters
    with open(os.path.join(os.getcwd(), TUNE_OUTPUT), 'r') as f:
        best_params = json.load(f)

    final_params = deepcopy(fixed_params)
    final_params.update(best_params)

    # Fit the model
    model = XGBClassifier(**final_params)
    model.fit(X_train,
              y_train,
              )

    # Save model
    if MODEL_SAVE:
        joblib.dump(model, os.path.join(os.getcwd(), MODEL_OUTPUT))

# Load model
else:
    model = joblib.load(os.path.join(os.getcwd(), MODEL_OUTPUT))

### Performance assessment

In [ ]:
# Load prob threshold
with open(os.path.join(os.getcwd(), TUNE_PROB_OUTPUT), 'r') as f:
    best_th = json.load(f)['best_th']

# Predict outcomes
y_pred_train = (model.predict_proba(X_train)[:, 1] >= best_th) * 1
y_pred_val = (model.predict_proba(X_val)[:, 1] >= best_th) * 1
y_pred_test = (model.predict_proba(X_test)[:, 1] >= best_th) * 1

# Train results
print('### Train results ###')
recall, precision, f1, acc,  cl_report = \
evaluate_predictions(true_values=y_train,
                     predictions=y_pred_train,
                     print_class_report=True)
print('f1:', f1)
print('recall:', recall)
print('precision:', precision)
print('accuracy:', acc)
print('\n')
results_cm_train = plot_confusion_matrix(y_train,
                                         y_pred_train,
                                         class_names=['Not Readmitted', 'Readmitted'],
                                         fig_size=(8,5),
                                         title='Confusion Matrix (XGBoost, Train Set)',
                                         colour_map='Greens'
                                         )

# Test results
print('### Validation results ###')
recall, precision, f1, acc,  cl_report = \
evaluate_predictions(true_values=y_val,
                     predictions=y_pred_val,
                     print_class_report=True)
print('f1:', f1)
print('recall:', recall)
print('precision:', precision)
print('accuracy:', acc)
print('\n')
results_cm_val = plot_confusion_matrix(y_val,
                                       y_pred_val,
                                       class_names=['Not Readmitted', 'Readmitted'],
                                       fig_size=(8,5),
                                       title='Confusion Matrix (XGBoost, Validation Set)',
                                       colour_map='Greens'
                                       )

# Test results
print('### Test results ###')
recall, precision, f1, acc,  cl_report = \
evaluate_predictions(true_values=y_test,
                     predictions=y_pred_test,
                     print_class_report=True)
print('f1:', f1)
print('recall:', recall)
print('precision:', precision)
print('accuracy:', acc)
print('\n')
results_cm_test = plot_confusion_matrix(y_test,
                                        y_pred_test,
                                        class_names=['Not Readmitted', 'Readmitted'],
                                        fig_size=(8,5),
                                        title='Confusion Matrix (XGBoost, Test Set)',
                                        colour_map='Greens'
                                        )

### Feature importance

In [ ]:
# Get feature importance
dict_imp = {
    'gain': model.get_booster().get_score(importance_type='gain'),
    'total_gain': model.get_booster().get_score(importance_type='total_gain'),
    'weight': model.get_booster().get_score(importance_type='weight'),
    'cover': model.get_booster().get_score(importance_type='cover'),
    'total_cover': model.get_booster().get_score(importance_type='total_cover'),
}
df_imp = pd.concat([pd.DataFrame(v, index=[k]).T for k, v in dict_imp.items()], axis=1).sort_values('total_gain', ascending=False)
df_imp.index = ['__'.join(c.split('__')[1:]) if '__' in c else c for c in df_imp.index]

# Plot feature importance
df_imp[['total_gain']].head(20).plot(kind='bar',
                                     title='Total Gain',
                                     )


### SHAP importance

In [ ]:
import shap

SAVE_PATH = Path('../../data/shap/xgboost_label_shap.csv')

# Diagnosis columns
cols_diag = [c for c in X_train if 'diag_' in c]

# Get SHAP importance
explainer = shap.TreeExplainer(model)
shap_train = explainer(X_train)

# Plot SHAP feature importance
feature_imp = shap_train.abs.mean(axis=0).values
df_shap = pd.DataFrame({'feature': X_train.columns, 'importance': feature_imp}).sort_values('importance', ascending=False)
df_shap.set_index('feature')[['importance']].head(20).plot(kind='bar',
                                                     title='SHAP Importance',
                                                     )

In [ ]:
# Save SHAP
if True:
    shap_diag = pd.DataFrame(shap_train.values, columns=shap_train.feature_names)[cols_diag]
    shap_diag.columns = [c.split('__')[1] + '_shap' for c in shap_diag.columns]
    shap_diag.to_csv(SAVE_PATH)
